In [31]:
#BallScrewKneeVectors_v3
#Code to compute knee torque when using a ball screw actuator with lever arm. 
#Vector analysis
#v3: adding a second lever arm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from vectoroperations import rotate2dsweep , triangle3rdpoint


In [32]:
plt.close('all')   

In [ ]:
#Vectors and constant parameters 
# %% Defining vectors for the first lever arm
O=np.array([0,0]) #Knee center
P0=np.array([0,0.05]) #Patella or lever arm middle position and length. 

OB=np.array([-0.05,0.01]).reshape(2,1) #Second pivot center
PB0=np.array([0,0.06]) #Second lever arm middle position vector



#Constants
H_bscrew=0.02 # Height of ball screw above knee center.
Forceballscrew=500 #Force applied by ball screw in N
L_pushrod=0.2 #Pushrod length from ballscrew
L_lever = np.linalg.norm(P0)   # Length of lever arm. 
LB_lever = np.linalg.norm(PB0)   # Length of second lever arm
LB_coupler=0.05  # coupler length 


AngP = np.linspace(40, 65, 26)  # N points, between the max and min angles. 
#AngP=np.atleast_1d(60)
P = rotate2dsweep(P0, AngP)   # Rotating Knee lever arm anglein degrees






In [34]:
# Finding position of second lever arm based on the position of the first lever arm. 4 bar linkage

#PB=PB0 #Test 1: not rotating the second lever arm
#PB = rotate2dsweep(PB0, AngP)   # Test 2: rotating second lever arm the same angle ast eh first. causes coupler length to change. 

PBvec = triangle3rdpoint(P, OB.reshape(2,1), LB_coupler, LB_lever)  # Finding the position of the second lever arm based on the position of the first lever arm. 4 bar linkage.
PB=PBvec-OB



In [35]:



#PBvec=OB.reshape(2,1)+PB #Vector to end of 2nd lever. old

#PB2Pvec=P-PBvec
PB2Pvec=P-PBvec #Vector from PB to P



# print('PB:', PB)
# print('PBvec:', PBvec)
# print('PB2Pvec:', PB2Pvec)




# Rotate2D(vector, angle in degrees CW +)
F = rotate2dsweep(P, 170)*2   # foot vector

Pvert = np.array([P[0], np.full_like(P[0], H_bscrew)]) # Vectors from knee to vertical projection of P onto line of action of ball screw.
Pup=Pvert-P #Vector from P to Pvert 


PBvert = np.array([PBvec[0], np.full_like(PB[0], H_bscrew)]) # Vectors from knee to vertical projection of second lever arm onto line of action of ball screw.
PBup=PBvert-(PBvec) #Vector from PB to Pvert 




In [36]:

# %% calculating point B (ball screw) using trianlge of B, P, and Pup

L_Pup=np.linalg.norm(Pup, axis=0) #Distance from P to Pvert
L_pushrodmin = np.sqrt(L_lever**2 + H_bscrew**2)  #checking pushrod length possibilities
L_bscrew=np.sqrt(L_pushrod**2-L_Pup**2) #Distance from Pup to B (ball screw)
BSmag = np.array([-L_bscrew, np.zeros_like(L_bscrew)]) #Ball screw vector
BS=BSmag+Pvert #Ball screw position in global coordinates
BSvec=Pvert-BS #Vector representing ball screw axis

#calculations for second lever arm
#LB_coupler = np.linalg.norm(PB2Pvec,axis=0)   # Length of coupler rod between the two lever arms. updated definition at top. 
LB_PBup=np.linalg.norm(PBup, axis=0) #Distance from PB to PBvert

LB_bscrew=np.sqrt(L_pushrod**2-LB_PBup**2) #Distance from Pup to B (ball screw)
LB_PB2Pvec=np.linalg.norm(PB2Pvec, axis=0) #Distance from PB to P. coupler length shoudl not change

BSBmag = np.array([-LB_bscrew, np.zeros_like(LB_bscrew)]) #Ball screw displacement vector
BSB=BSBmag+PBvert #Ball screw position in global coordinates
BSBvec=PBvert-BSB #Vector representing ball screw axis



# print('P0:', P0)
# print('P:', P)  
# print('Pup:', Pup)
# print('F:', F)
# print('BS:', BS)
# print('L_Pup:', L_Pup)
# print('L_bscrew:', L_bscrew)
# print('L_pushrodmin:', L_pushrodmin)
print('LB_PB2Pvec:', LB_PB2Pvec)





LB_PB2Pvec: [0.05]


In [ ]:
# %% Torque calculations


Pushrodvec=P - BS
Pushrodvec_unit = Pushrodvec / np.linalg.norm(Pushrodvec, axis=0)
BSvec_unit = BSvec / np.linalg.norm(BSvec, axis=0)
Fpushrod = np.sum(BSvec_unit * Pushrodvec_unit, axis=0)*Forceballscrew  # finding componnt of ballscrew force acting along pushrod
torque_O = Fpushrod * np.cross(P, Pushrodvec_unit, axis=0) # Torque at knee center O




PushrodvecPB=PBvec - BSB
#PushrodvecPB=PB - BSB # wrong

PushrodvecPB_unit = PushrodvecPB / np.linalg.norm(PushrodvecPB, axis=0)
BSBvec_unit = BSBvec / np.linalg.norm(BSBvec, axis=0)
FpushrodPB = np.sum(BSBvec_unit * PushrodvecPB_unit, axis=0)*Forceballscrew  # finding componnt of ballscrew force acting along pushrod

# checking angles between vectors
# dot_val = np.sum(BSBvec * PushrodvecPB, axis=0)
# cross_val = BSBvec[0]*PushrodvecPB[1] - BSBvec[1]*PushrodvecPB[0]   # 2D "cross product" (scalar)
# Ang_rad = np.arctan2(cross_val, dot_val)
# Ang_deg = np.degrees(Ang_rad)



PB2Pvec_unit = PB2Pvec / LB_PB2Pvec                  # unit vector along coupler rod
Fcoupler = np.sum(FpushrodPB * PB2Pvec_unit, axis=0)  # component of pushrod force along coupler. 

torque_O2 = Fcoupler * np.cross(P, PB2Pvec_unit, axis=0) # Torque at knee center O due to coupler force




# display(df1)
# display(df2)


# df = pd.DataFrame({
#     'Angle_deg': AngP,
#     'Torque_Nm': torque_O
# })
# df.to_csv('Ballscrewknee_torque_vs_angle.csv', index=False)




C:\Users\smohamedthangal\AppData\Local\Temp\ipykernel_33720\1415444703.py:8: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  torque_O = Fpushrod * np.cross(P, Pushrodvec_unit, axis=0) # Torque at knee center O
C:\Users\smohamedthangal\AppData\Local\Temp\ipykernel_33720\1415444703.py:31: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  torque_O2 = Fcoupler * np.cross(P, PB2Pvec_unit, axis=0) # Torque at knee center O due to coupler force


In [38]:
%matplotlib qt
# Use widget in Syzygy to get an interactive window. qt dont work in syzygy
#%matplotlib widget


In [39]:
# %% Graphing

# vectors quiver plot
fig = plt.figure()
fig.canvas.manager.set_window_title('Ballscrewknee')
ax = plt.gca()

Ox = np.full_like(P[0], O[0])# creates an array of the same shape as P[0] filled with O[0]
Oy = np.full_like(P[0], O[1])

plt.quiver(Ox, Oy, P[0], P[1], angles='xy', scale_units='xy', scale=1,width=0.002, color='r', label='P')
plt.quiver(Ox, Oy, F[0], F[1], angles='xy', scale_units='xy', scale=1,width=0.002, color='darkred', label='F')
plt.quiver(P[0], P[1], Pup[0], Pup[1], angles='xy', scale_units='xy', scale=1, width=0.002, color='0.9', label='Pup')
plt.quiver(BS[0], BS[1], P[0]-BS[0], P[1]-BS[1], angles='xy', scale_units='xy', scale=1,width=0.002, color='0.9', label='Pushrod')
plt.quiver(BS[0], BS[1], Pvert[0]-BS[0], Pvert[1]-BS[1], angles='xy', scale_units='xy', scale=1,width=0.002, color='0.9', label='Ballscrew axis')


OBx = np.full_like(PB[0], OB[0])# creates an array of the same shape as P[0] filled with O[0]
OBy = np.full_like(PB[0], OB[1])

plt.quiver(O[0], O[1], OB[0], OB[1], angles='xy', scale_units='xy', scale=1, width=0.005, color='k', label='OB')
plt.quiver(OBx, OBy, PB[0], PB[1], angles='xy', scale_units='xy', scale=1, width=0.002, color='orange', label='PB0')
plt.quiver(PBvec[0], PBvec[1], PB2Pvec[0], PB2Pvec[1], angles='xy', scale_units='xy', scale=1, width=0.002, color='g', label='PB2P')

plt.quiver(PBvec[0], PBvec[1], PBup[0], PBup[1], angles='xy', scale_units='xy', scale=1, width=0.002, color='gray', label='PBup')
plt.quiver(BSB[0], BSB[1], PBvec[0]-BSB[0], PBvec[1]-BSB[1], angles='xy', scale_units='xy', scale=1,width=0.002, color='m', label='PushrodB')
plt.quiver(BSB[0], BSB[1], PBvert[0]-BSB[0], PBvert[1]-BSB[1], angles='xy', scale_units='xy', scale=1,width=0.002, color='gray', label='Ballscrew axis')



circle = patches.Circle((0, 0), radius=L_lever, edgecolor='black', facecolor='none', linewidth=0.3)
circle2 = patches.Circle((OB[0, 0], OB[1, 0]), radius=LB_lever, edgecolor='green', facecolor='none', linewidth=0.2)

ax.add_patch(circle)
ax.add_patch(circle2)

label_offset = 0.005  # small offset so text doesn't sit directly on the point

ax.text(O[0] + label_offset, O[1] + label_offset, 'O', fontsize=9, color='k')
ax.text(OB[0, 0] + label_offset, OB[1, 0] + label_offset, 'OB', fontsize=9, color='k')


if P.ndim >1 and P.shape[1] == 1:
    # runs if P is a single vector only 
    points = {
        'O':   (O[0], O[1], 'k'),
        'OB':  (OB[0, 0], OB[1, 0], 'k'),
        'Patella':   (P[0, 0], P[1, 0], 'r'),
        'Ballscrew': (BSB[0, 0], BSB[1, 0], 'gray'),
        'PB':  (PBvec[0, 0], PBvec[1, 0], 'orange'),
    }
    for name, (x, y, c) in points.items():
        ax.text(x + label_offset, y + label_offset, name, fontsize=9, color=c)




siz = 0.20
plt.xlim(-siz*2, siz*2)   # X limits
plt.ylim(-siz, siz)   # quiver doesn't auto-scale the view the way plot() does
ax.set_aspect('equal', adjustable='box')
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.grid(True)
plt.legend()
plt.show()

# torque vs angle plot
fig = plt.figure()
fig.canvas.manager.set_window_title('Torque vs angle')
ax = plt.gca()
plt.plot(AngP, torque_O, linewidth=2, color='k', marker='.', linestyle='-')
plt.plot(AngP, torque_O2, linewidth=2, color='r', marker='.', linestyle='-')
#ax.set_aspect('equal', adjustable='box')
plt.grid(True)
plt.xlabel('Angle (degrees)')
plt.ylabel('Torque (N.m)')
plt.legend(['Single lever', 'two levers'])
plt.show()



# Force vs angle plot
fig = plt.figure()
fig.canvas.manager.set_window_title('Force vs angle')
ax = plt.gca()
plt.plot(AngP, FpushrodPB, linewidth=2, color='g', marker='.', linestyle='-')
plt.plot(AngP, Fcoupler, linewidth=2, color='m', marker='.', linestyle='-')
#ax.set_aspect('equal', adjustable='box')
plt.grid(True)
plt.xlabel('Angle (degrees)')
plt.ylabel('Force (N)')
plt.legend(['FpushrodPB', 'Fcoupler'])
plt.show()

# # Use block in VSC to allow scrpit compeltion. not needed in Spyder. 
# plt.show(block=False)
# input("Press Enter to continue...")
# plt.close('all')





In [40]:

# %%

%whos

Variable            Type        Data/Info
-----------------------------------------
AngP                ndarray     1: 1 elems, type `int64`, 8 bytes
BS                  ndarray     2x1: 2 elems, type `float64`, 16 bytes
BSB                 ndarray     2x1: 2 elems, type `float64`, 16 bytes
BSBmag              ndarray     2x1: 2 elems, type `float64`, 16 bytes
BSBvec              ndarray     2x1: 2 elems, type `float64`, 16 bytes
BSBvec_unit         ndarray     2x1: 2 elems, type `float64`, 16 bytes
BSmag               ndarray     2x1: 2 elems, type `float64`, 16 bytes
BSvec               ndarray     2x1: 2 elems, type `float64`, 16 bytes
BSvec_unit          ndarray     2x1: 2 elems, type `float64`, 16 bytes
F                   ndarray     2x1: 2 elems, type `float64`, 16 bytes
Fcoupler            ndarray     1: 1 elems, type `float64`, 8 bytes
Forceballscrew      int         500
Fpushrod            ndarray     1: 1 elems, type `float64`, 8 bytes
FpushrodPB          ndarray     1: 1 el